# HPC-PatentTrace step 6: from the crawler output to the released dataset

This notebook turns `HPC_15to25_v2.csv`, the output of step 5 of this repository, into the HPC-PatentTrace data release, the dataset of the data descriptor *High-performance computing patent dataset with full text, structured summaries and semantic embeddings*. Each section below is one step; its code is in `colab/hpc_patenttrace/steps/`, and the notebook shows only the figures and tables that the step produces.

| Section | Step | Records (released run) | GPU |
|---|---|---|---|
| 1 | Setup: packages, configuration, checks, reuse of generated layers | | |
| 2 | Homonym screening: remove records in which HPC has another meaning | 21,947 → 16,434 | no |
| 3 | Corpus rules: five complete sections, expanded phrase in the text | 16,434 → 7,158 → 6,535 | no |
| 4 | Facets: three short summaries per record (Llama-3.1-8B-Instruct) | 6,535 | yes |
| 5 | Summary layers: synthesized, direct-brief, direct-full (Qwen2.5-14B-Instruct) | 6,535 | yes |
| 6 | SPECTER2 embeddings, clusters and merged topics | 6,535 | partly |
| 7 | Ownership evidence and usage examples | 6,535 | no |
| 8 | Technical validation of the summary layers | 6,535 | partly |
| 9 | Release files and cost table | | no |
| 10 | Run report | | no |

**How to run on Google Colab**

1. Put the input files in one folder of Google Drive (by default `MyDrive/HPC-PatentTrace/input/`): `HPC_15to25_v2.csv`, `patents.dta` of the [USPTO Patent Litigation Docket Reports](https://www.uspto.gov/ip-policy/economic-research/research-datasets/patent-litigation-docket-reports-data).
2. Choose a GPU runtime and check the Parameters cell.
3. Run all cells (Runtime → Run all). Outputs go to `output/full/` of the same folder.

With `REUSE_FROM` empty, the facets and the three summary layers are generated, which took about 20 hours on one A100 80 GB GPU for the released run. With `REUSE_FROM` set to the output folder of an earlier full run, no generation model is loaded and every other step is computed again; we estimate 1 to 1.5 hours on one L4 or A100 GPU. Every long step writes a checkpoint, so an interrupted run continues where it stopped.

## Parameters

Fill in the folder and the input files; nothing else in the notebook needs to be changed. A relative path is read inside `DATA_DIR`. The `sample` profile runs every step on 400 input records drawn at random; on a GPU with less than 32 GB (or without a GPU) it uses the small model Qwen2.5-0.5B-Instruct for both generation steps, which only tests that the code runs.

In [ ]:
#@title Parameters
#@markdown **Folder** that holds the input files and receives the outputs (on Colab, a folder of Google Drive).
DATA_DIR = "/content/drive/MyDrive/HPC-PatentTrace"  #@param {type:"string"}
#@markdown **Input files**, relative to DATA_DIR or absolute.
INPUT_CSV = "input/HPC_15to25_v2.csv"  #@param {type:"string"}
LITIGATION_FILE = "input/patents.dta"  #@param {type:"string"}
#@markdown **Reuse** the generated layers of an earlier full run: the output folder of that run, for example `output/full` (this run then needs another OUTPUT_DIR). Leave empty to generate them (about 20 hours on one A100 80 GB).
REUSE_FROM = ""  #@param {type:"string"}
#@markdown **Output** folder, relative to DATA_DIR or absolute; the run writes to OUTPUT_DIR/PROFILE.
OUTPUT_DIR = "output"  #@param {type:"string"}
#@markdown **Profile**: `full` runs the whole input; `sample` runs every step on 400 records drawn at random.
PROFILE = "full"  #@param ["full", "sample"]
SAMPLE_MODELS = "auto"  #@param ["auto", "real", "small"]
#@markdown **Hugging Face token**, needed only to generate the facets with Llama-3.1-8B-Instruct; a Colab secret named HF_TOKEN also works.
HF_TOKEN = ""  #@param {type:"string"}
#@markdown Print the log of each step in the notebook (it is always written to OUTPUT_DIR/PROFILE/logs/).
SHOW_LOG = False  #@param {type:"boolean"}

## 1  Setup

Cell 1.1 installs the packages that Colab does not have and copies the code of this repository (`REPO_REF` selects a branch or tag). Cell 1.2 mounts Google Drive, checks the input files and the packages, and, when `REUSE_FROM` names the output folder of an earlier run, copies its generated layers into this run as checkpoints.

Writes: `logs/`, `cell_times.csv`, `runtime_log.csv`, `reuse_manifest.csv`

In [ ]:
#@title 1.1 · Install the packages and get the code of this repository
#@markdown On Google Colab this cell installs the packages and copies the code of the repository; on a local computer, start Jupyter in `colab/` of the repository.
import os, sys
REPO_URL = "https://github.com/Taco-Yun/hpc-patent-dataset"
REPO_REF = "main"  #@param {type:"string"}
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "transformers>=4.45.0" accelerate sentencepiece adapters umap-learn hdbscan bert-score minisom rouge-score sentence-transformers pyarrow
    %pip install -q bitsandbytes
    if not os.path.isdir("/content/hpc-patent-dataset"):
        !git clone -q --depth 1 --branch {REPO_REF} {REPO_URL} /content/hpc-patent-dataset
    sys.path.insert(0, "/content/hpc-patent-dataset/colab")
else:
    sys.path.insert(0, os.getcwd())
from hpc_patenttrace import run_step, show

In [ ]:
#@title 1.2 · Prepare the run
if IN_COLAB and DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
run_step("s1_setup")
show(RUN_SETTINGS)

## 2  Homonym screening

HPC means high-performance computing in computing patents, but hydroxypropyl cellulose in pharmaceutical patents and high-pressure compressor in turbine patents. A record is removed only when its own text shows another sense: the authors define the abbreviation with another long form, or the full phrase of another sense occurs in the text. The class code is reported but never used to decide.

Writes: `HPC_15to25_clean.csv` (candidates), `provenance/removed_homonym.csv`, `provenance/candidate_ids.csv`, figures 2.1–2.3

In [ ]:
#@title 2 · Homonym screening
run_step("s2_homonym")
show(removed_tab.reset_index(), "Removed records by sense and evidence")

## 3  Corpus rules

The completeness filter keeps a candidate when its title, abstract, background, detailed description and claims all have text. The inclusion rule keeps a record when its text contains the expanded phrase `high performance comput*`. Records that pass the first rule but not the second are released in full in `excluded_no_expansion.csv`.

Writes: `released_corpus_ids.csv`, `evidence_flags.csv`, `excluded_no_expansion.csv`, `litigation_family.csv`, `tables/`, figure 3.1

In [ ]:
#@title 3 · Corpus rules
run_step("s3_corpus")
show("tables/funnel_from_crawler_output.csv", "Records at each step")
show("differential_exclusion.csv", "Inclusion rule by class section")

## 4  Facets

Llama-3.1-8B-Instruct (4-bit NF4) summarises the background (1,500 tokens), the summary input (1,500) and the detailed description (2,000) of each record in a few sentences each, with greedy decoding. The model is loaded only when facets are still missing.

Writes: `stage1_checkpoint.csv`, `stage1_summaries.csv`, figure 4.1

In [ ]:
#@title 4 · Facets
run_step("s4_facets")

## 5  Summary layers

The three layers share the last generation call: Qwen2.5-14B-Instruct in bf16, greedy decoding, at most 300 new tokens, five to seven sentences. The **synthesized** layer is written from the title, the abstract and the facets after a reasoning call.

Writes: `stage2_checkpoint.csv`, `patent_full_pipeline.csv`

In [ ]:
#@title 5.1 · Synthesized summaries
run_step("s5_synthesized")

**direct-brief** and **direct-full** use one prompt over the sections; direct-brief reads 800, 400 and 800 tokens, direct-full the facet budgets of 1,500, 1,500 and 2,000 tokens.

Writes: `baseline_direct.csv`, `baseline_direct_matched.csv`

In [ ]:
#@title 5.2 · direct-brief and direct-full summaries
run_step("s5_direct")

## 6  Embeddings, clusters and topics

SPECTER2 with the proximity adapter encodes the title and the synthesized summary of each record. UMAP reduces the vectors to three dimensions, HDBSCAN clusters them (`min_cluster_size=30`, `min_samples=10`, the only setting of the six-point sweep with a DBCV above 0.3), and the clusters are merged into 25 topics. With reused layers, the clusters are computed again on the released coordinates and compared with the released assignment, which is kept.

Writes: `cluster_results.csv`, `topic_summary.csv`, `merge_map_worksheet.csv`, `hdbscan_param_sweep.csv`, `embeddings/`

In [ ]:
#@title 6 · Embeddings, clusters and topics
run_step("s6_topics")
show("hdbscan_param_sweep.csv", "HDBSCAN settings")
show("topic_summary.csv", "Merged topics")

## 7  Ownership evidence and usage examples

The latest assignment record of each patent is classified with nine ordered rules; `effective_assignee` keeps the original assignee when no record that transfers ownership supports a change. The usage examples show assignee portfolios, topic trends, citations and the agreement of the topics with the patent classification.

Writes: `ownership_events.csv`, `usage_*.csv`, `topic_classifier_benchmark.csv`, figures 7.1–7.9 (7.1, 7.2, 7.4 and 7.5 are interactive HTML files)

In [ ]:
#@title 7 · Ownership evidence and usage examples
run_step("s7_ownership_usage")
show("usage_assignee_before_after.csv", "Ten largest assignees before and after the ownership evidence")

## 8  Technical validation

Each check asks whether the data are wrong, not whether one layer beats another: coverage of the token budgets, terms kept, content beyond the abstract, the pairing check against the shifted abstract, the human-audit sample, CPC and assignee classification from each representation, perplexity, similarity to the abstract and BERTScore.

Writes: `summary_metrics.csv`, `pairing_integrity_flags.csv`, `added_value_beyond_abstract.csv`, `ipc_downstream_comparison.csv`, `ppl_summaries.csv` and the other validation tables

In [ ]:
#@title 8 · Technical validation
run_step("s8_validation")
show("summary_metrics.csv", "Summary layers against the abstract")
show("ipc_downstream_comparison.csv", "CPC classification from each representation")

## 9  Release files

The prompts, term lists and settings are written from the objects of this run, the cost table is completed, and the release files are written to `release/`. Part 9.4 then replaces ipc by cpc in every file and column name of `release/`, because the class codes are CPC codes, and checks each converted file against the original: same rows, identifiers, class codes and values. The working files keep their names.

Writes: `release/`, `release/MANIFEST_release.csv`, `release/NAME_MAPPING_ipc_to_cpc.csv`, `runtime_log_release.csv`

In [ ]:
#@title 9 · Release files
run_step("s9_release")
show("release/MANIFEST_release.csv", "Files of the release", rows=60)
show(NAME_CHECK, "Release names: files converted from ipc to cpc and checked")

## 10  Run report

The report gives the environment, the wall time of each section and, for the full profile, seventeen key numbers of this run next to those of the released run (2026-10-11). The counts of sections 2 and 3 follow from rules and must be the same; the other numbers depend on the generated layers and are the same when the layers of the released run are reused.

In [ ]:
#@title 10 · Run report
run_step("s10_report")
show(key_df, "Key numbers")
show(by_section.reset_index(), "Wall time by section")
show(env_df, "Environment")

### Notes

- A full generation on other hardware or library versions can give a different text even with greedy decoding.
- The class codes come from the CPC paths of the Google Patents pages. In `release/` the names say cpc (`main_cpc`, `main_cpc_4`, `main_cpc_3`, `cpc_description`, `cpc_downstream_comparison.csv`, ...); the crawler output and the working files keep the historical names with ipc, and `release/NAME_MAPPING_ipc_to_cpc.csv` lists every old and new name.
- `reproduce_figures_tables.ipynb` draws the figures and tables of the data descriptor from the output folder of a full run of this notebook.